Now I will utilise sklearn to perform Ridge regression on the data to estimate how each athlete does on a particular boulder. In terms of scoring, we use the % of the best score as our metric of comparison

I've chosen to use Ridge regression because, whilst I've compiled lots of data, it is still a relatively small data set. So I would like to minimise the variance in my coefficients. Ridge regression was selected over Lasso regression because Lasso regression may assign a value of 0 to some coefficients, however, I believe all the features of the data I've collected are important so I would like to avoid this.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

Semifinals = pd.read_csv("SemiFinalists.csv")
style = pd.read_csv("BoulderStyle.csv")
style["Boulder_Num"] = style["Boulder_Num"].astype(int)
BoulderResults = pd.read_csv("BoulderResults.csv")

In [ ]:
events_2026 = ["KEQ26", "BER26", "MAD26", "PRA26", "IBK26"]
merged = pd.read_csv("Boulder_Results_with_Style.csv")


attempts = merged.copy()
affinity = pd.read_csv("Athlete_Style_Affinity_Shrunk.csv")


MLdata = attempts.merge(affinity, on="Athlete_ID", how="inner")  # Combine data together to use sklearn

style_labs = ["Slab", "Coordination", "Power", "Compression", "Dynamic", "Press"]
affinity_cols = [f"{s}_Affinity_Shrunk" for s in style_labs]
stats_cols = style_labs + affinity_cols

# I want to train the model using previous data to test on the 2026 data
events_2026 = ["KEQ26", "BER26", "MAD26", "PRA26", "IBK26"]
train_df = MLdata[~MLdata["Event_ID"].isin(events_2026)].copy()
test_df = MLdata[MLdata["Event_ID"].isin(events_2026)]

X_train, Y_train = train_df[stats_cols], train_df["Pct_Of_Best"] # Training data
X_test, Y_test = test_df[stats_cols], test_df["Pct_Of_Best"]     # Testing data

# -----------------------------------------------------------------------------

# I will be using Ridge regression for this model with coefficient 1

model = Ridge(alpha=1.0)
model.fit(X_train, Y_train)

predictions = model.predict(X_test)

# -----------------------------------------------------------------------------

print("R^2 on 2026 holdout:", round(r2_score(Y_test, predictions), 3)) # Obtain R^2 coefficient
print("MAE on 2026 holdout:", round(mean_absolute_error(Y_test, predictions), 3)) # Obtained MAE

coef_table = pd.DataFrame({"Feature": stats_cols, "Coefficient": model.coef_}).sort_values("Coefficient", ascending=False)
print(coef_table.to_string(index = False))


FileNotFoundError: [Errno 2] No such file or directory: 'Athlete_Style_Affinity_Shrunk.csv'

We see that the R^2 value is only 0.121 meaning about 88% of the variance is not considered by the model. As a result, it would be worth considering changing the model. Furthermore, the mean absolute error is 0.291. Since we are basing off of percentage of the maximum, this is not exactly ideal.

Furthermore, the negative coefficients obtained is counter intuitive to what we would expect, that is, it is showing even if an athlete has a high affinity to a style, if the boulder is in that style, it does not particularly give them a higher percentage of the best score.

Below I consider a model with an interaction term

In [ ]:
# Model using interaction term

for s in style_labs:
    MLdata[f"{s}_Interaction"] = MLdata[s] * MLdata[f"{s}_Affinity_Shrunk"]

interaction_cols = [f"{s}_Interaction" for s in style_labs]  # Including interaction terms
stats_cols = style_labs + affinity_cols + interaction_cols

train_df = MLdata[~MLdata["Event_ID"].isin(events_2026)].copy()
test_df = MLdata[MLdata["Event_ID"].isin(events_2026)]

X_train, Y_train = train_df[stats_cols], train_df["Pct_Of_Best"]
X_test, Y_test = test_df[stats_cols], test_df["Pct_Of_Best"]

model = Ridge(alpha=1.0)
model.fit(X_train, Y_train)

predictions = model.predict(X_test)

print("R^2 on 2026 holdout:", round(r2_score(Y_test, predictions), 3))
print("MAE on 2026 holdout:", round(mean_absolute_error(Y_test, predictions), 3))

coef_table = pd.DataFrame({"Feature": stats_cols, "Coefficient": model.coef_}).sort_values("Coefficient", ascending=False)
print(coef_table.to_string(index=False))

We report the R^2 value as 0.201. Clearly there is an improvement by adding the interaction term and the model with the interaction terms explain more of the variance. The MAE value is also smaller.

The reason for the unintuitive coefficients is likely due to high correlation between these styles. Whilst it is true that athletes have their own particular style, boulders can include multiple styles at once, hence, the data and our model may not be able to clearly make this distinction. As well as this, athletes that score well are typically stronger in all styles too.


# Backtesting against the finalists

Using the model we've trained above, I want to test how accurate our model is at predicting the 2026 finalists

In [ ]:
# Testing how well our Ridge regression model holds up against real data

def predict_finalists_for_event(event_id, roster, style_df, affinity_df, model, style_labs, affinity_cols, stats_cols, top_n = 8):
  # roster is the athletes. top_n = 8 because finals has 8 athletes
  # We grab the actual style of the boulders from the semi finals
  sf_boulders = style_df[(style_df["Event_ID"] == event_id) & (style_df["Round"] == "Semifinal")]

  # Build rows for each (athlete,boulder) pair to store the score per athlete per boulder
  rows = []
  for _, boulder in sf_boulders.iterrows():
    for athlete_id in roster:
      row = {"Athlete_ID" : athlete_id}
      for s in style_labs:
        row[s] = boulder[s]             # creating a dictionary to store each athlete alongside each boulder and its style
      rows.append(row)
  pred_df = pd.DataFrame(rows)

  # Now I have to also add the affinity scores
  pred_df = pred_df.merge(affinity_df, on="Athlete_ID", how="left")

  # Add a column to hold the interaction terms for our model
  for s in style_labs:
      pred_df[f"{s}_Interaction"] = pred_df[s] * pred_df[f"{s}_Affinity_Shrunk"]

  # Now predict the score for each semi finals boulder
  pred_df["Predicted_Score"] = model.predict(pred_df[stats_cols])

  # Sum the predicted score for each athlete across the round
  total_score = pred_df.groupby("Athlete_ID")["Predicted_Score"].sum().reset_index()
  total_score = total_score.sort_values("Predicted_Score", ascending=False).head(top_n)

  # Now we return the top 8 athletes because these are our finalists
  return total_score.head(top_n)["Athlete_ID"].tolist()


def actual_finalists_for_event(event_id, results_df):  # Gives the finalists for the event
    return set(results_df[(results_df["Event_ID"] == event_id) &
                           (results_df["Round"] == "Final")]["Athlete_ID"])


def compare_finalist_pool(predicted, event_id, results_df): # Compare prediction vs actual results
  actual = actual_finalists_for_event(event_id, results_df)
  predicted_set = set(predicted)
  overlap = predicted_set & actual  # Contains the overlapping finalists in our predicted set and actual

  return {  # Results in a dictionary
        "Event_ID": event_id,
        "Overlap": len(overlap),
        "Precision": round(len(overlap) / len(predicted_set), 3) if predicted_set else None,
        "Recall": round(len(overlap) / len(actual), 3) if actual else None,
    }

# -----------------------------------------------------------------------
# Now to run it against the 2026 events using the Ridge regression model with interaction
# Reusing our previously assigned data
affinity_cols = [f"{s}_Affinity_Shrunk" for s in style_labs]
interaction_cols = [f"{s}_Interaction" for s in style_labs]
stats_cols = style_labs + affinity_cols + interaction_cols

roster = Semifinals["Athlete_ID"].tolist() # Potential finalists

finalist_rows = []  # Contain the comparison between real and predicted finalists
for eid in events_2026:
    predicted = predict_finalists_for_event(eid, roster, style, affinity, model, style_labs, affinity_cols, stats_cols)
    finalist_rows.append(compare_finalist_pool(predicted, eid, BoulderResults))

FinalistComparisondf = pd.DataFrame(finalist_rows)
print(FinalistComparisondf[["Event_ID", "Overlap", "Precision", "Recall"]])

total_overlap = FinalistComparisondf["Overlap"].sum()

print("\nOverall precision across all 2026 finals:", round(total_overlap / (8*5), 3))
print("Overall recall across all 2026 finals:", round(total_overlap / (8*5), 3))


We see we have a 45% precision. This is pretty good. Considering we can only have 8 finalists, this means we typically correctly predict about half of the athletes entering. Considering that, in the sport of climbing, it is not typical for the finals to always be the same (the same 8 athletes to dominant in every style), this is a normal result.